In [12]:
from nltk.corpus.reader import CorpusReader
from nltk.corpus.reader.api import CategorizedCorpusReader
import nltk
import os
import json

corpus_root = 'data/corpus'
nltk.download('punkt_tab')

abs_corpus_root = os.path.abspath(corpus_root)
if abs_corpus_root not in nltk.data.path:
    nltk.data.path.append(abs_corpus_root)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Niwei\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [ ]:
class AmazonCorpusReader(CategorizedCorpusReader, CorpusReader):
    def __init__(self, root='data/corpus', fileids=r'.*\.json', encoding='utf8', **kwargs):
        CategorizedCorpusReader.__init__(self, kwargs)
        CorpusReader.__init__(self, root, fileids, encoding=encoding)

    def _resolve(self, fileids, categories):
        # Combines category-based filtering with standard fileid lookup
        if fileids is not None:
            if isinstance(fileids, str):
                fileids = [fileids]
            return fileids
        return self.fileids(categories)

    def _parse_jsonl(self, stream):
        for line in stream:
            line = line.strip()
            if line:
                yield json.loads(line)

    def reviews(self, fileids=None, categories=None):
        fileids = self._resolve(fileids, categories)
        for review in fileids:
            with self.open(review) as f:
                for review in self._parse_jsonl(f):
                    yield review

    def ratings(self, fileids=None, categories=None):
        for review in self.reviews(fileids=fileids, categories=categories):
            yield review['rating']

    def texts(self, fileids=None, categories=None):
        for review in self.reviews(fileids=fileids, categories=categories):
            yield review['text']

    def paragraphs(self, fileids=None, categories=None):
        for text in self.reviews(fileids=fileids, categories=categories):
            yield [p.strip() for p in text.split('\n\n') if p.strip()]

    def sentences(self, fileids=None, categories=None):
        for text in self.texts(fileids=fileids, categories=categories):
            yield nltk.sent_tokenize(text)

    def words(self, fileids=None, categories=None):
        for text in self.texts(fileids=fileids, categories=categories):
            yield nltk.word_tokenize(text)

In [ ]:
cat_pattern = r"(negative|neutral|positive)/.*"
reader = AmazonCorpusReader(root='data/corpus/', fileids=r'.*\.json', cat_pattern=cat_pattern)

print("Categories:", reader.categories())
print("File IDs:", reader.fileids())
print("File IDs for 'positive':", reader.fileids(categories='positive'))
print ("content:", reader.raw('positive/test_0000002.json'))
print("Ratings:", reader.ratings(fileids='positive/test_0000002.json'))
for r in reader.ratings(fileids='positive/test_0000002.json'):
    print("Rating:", r)
print("Text:", reader.texts(fileids='positive/test_0000002.json'))
for t in reader.texts(fileids='positive/test_0000002.json'):
    print("Text:", t)
print("Word:", reader.words(fileids='positive/test_0000002.json'))
for w in reader.words(fileids='positive/test_0000002.json'):
    print("Words:", w)

Categories: ['negative', 'neutral', 'positive']
File IDs: ['negative/test_0000001.json', 'negative/test_0000002.json', 'negative/test_0000003.json', 'negative/test_0000004.json', 'negative/test_0000005.json', 'negative/test_0000006.json', 'negative/test_0000007.json', 'negative/test_0000008.json', 'negative/test_0000009.json', 'negative/test_0000010.json', 'negative/test_0000011.json', 'negative/test_0000012.json', 'negative/test_0000013.json', 'negative/test_0000014.json', 'negative/test_0000015.json', 'negative/test_0000016.json', 'negative/test_0000017.json', 'negative/test_0000018.json', 'negative/test_0000019.json', 'negative/test_0000020.json', 'negative/test_0000021.json', 'negative/test_0000022.json', 'negative/test_0000023.json', 'negative/test_0000024.json', 'negative/test_0000025.json', 'negative/test_0000026.json', 'negative/test_0000027.json', 'negative/test_0000028.json', 'negative/test_0000029.json', 'negative/test_0000030.json', 'negative/test_0000031.json', 'negative/t

In [ ]:
reader.raw('positive/test_0000002.json')

'{"rating": 5.0, "title": "Coctique", "text": "I loved every thing and could use it all. Things were in it that I would never order Coctique", "asin": "B07NVL6TJG", "parent_asin": "B07NVKNVNM", "user_id": "AE6P2YJ6FKX332MD56GPJFSHXNJQ", "verified_purchase": true, "helpful_vote": 0, "label": "positive", "source": "test"}'